# Tâche 1 - Modèles de langue N-grammes

Cette tâche consiste à construire, avec NLTK, des modèles de langue Ngrammes à partir de titres de chansons et de les utiliser pour accomplir 2 sous-tâches qui sont décrites dans l'énoncé du travail pratique. 

Les fichiers à utiliser pour cette tâche :
- *t1_titres_chansons.txt*: il contient une liste de titres de chansons, un par ligne de texte. Vous utilisez ce fichier pour la construction des modèles de langues N-grammes. 
- *t1_estimation_test.json*: il contient des listes de candidats, chaque liste contenant un seul titre de chanson connu parmi les candidats. 
- *t1_generation_test.json*: il contient des titres incomplets se terminant par des mots masqués.  

Consignes: 
- Ne pas modifier les fonctions *load_titles* et *load_tests*.
- Utilisez la variable *models* pour conserver les modèles après entraînement. 
- Ne pas modifier la signature des fonctions *train_models* et *complete_title*.
- Des modifications aux signatures de fonctions entraîneront des pénalités dans la correction. 
- Vous pouvez ajouter des cellules au *notebook* et ajouter toutes les fonctions utilitaires que vous voulez. 

## Section 1 - Lecture des fichiers de données (titres de chansons et tests)

On définit ici 2 fonctions pour lire les fichiers d'entraînement et de test. 

In [1]:
import json


titres_fn = "./data/t1_titres_chansons.txt"    
test_estimation_fn = './data/t1_estimation_test.json'  
test_generation_fn = './data/t1_generation_test.json'  


def load_titles(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()
    return [x.strip() for x in raw_lines]


def load_tests(filename):
    with open(filename, 'r', encoding='utf-8') as fp:
        test_data = json.load(fp)
    return test_data

In [2]:
titles = load_titles(titres_fn)

Quelques informations à propos des titres: 

In [3]:
print("Nombre de titres pour l'entraînement: {}".format(len(titles)))

Nombre de titres pour l'entraînement: 572


On monte également les 2 fichiers de test pour visualiser leur contenu. 

In [4]:
import pandas as pd

def get_dataframe(tests):
    return pd.DataFrame.from_dict(tests, orient='columns', dtype=None, columns=None)

In [5]:
tests_estimation = load_tests(test_estimation_fn)

In [6]:
df = get_dataframe(tests_estimation)
pd.set_option("display.max_colwidth", 120)
display(df)

,Candidates,Song
0,"[i want to hold and love, i want to hold something, i want to hold your hand]",i want to hold your hand
1,"[let it be, let me be, let it roll]",let it be
2,"[i want to know the world, i want to love the way you lie, i want to know what love is]",i want to know what love is
3,"[i will survive, i will be there, i will follow and wait]",i will survive
4,"[i will wait, i will eat, i will break free]",i will wait
5,"[i gotta feeling, i gotta rock you, i gotta a feeling to rock you]",i gotta feeling
6,"[i want to break free, i want to break the mold, i want to break my mold free]",i want to break free
7,"[we will rock you, i will rock you, she will rock you]",we will rock you
8,"[i kissed a girl, i kissed a boy, i kissed nobody]",i kissed a girl
9,"[i believe i can fly, i believe i can rock, i believe i can wake up]",i believe i can fly


In [7]:
tests_generation = load_tests(test_generation_fn)

df = get_dataframe(tests_generation)
pd.set_option("display.max_colwidth", 100)
display(df)

,Masked,Song
0,love ***,love shack
1,i want to hold your ***,i want to hold your hand
2,i want to hold *** ***,i want to hold your hand
3,i want *** *** *** ***,i want to hold your hand
4,let it ***,let it be
5,i will ***,i will survive
6,i gotta ***,i gotta feeling
7,i want to break ***,i want to break free
8,i want to *** ***,i want to break free
9,i want *** *** ***,i want to break free


## Section 2 - Construction des modèles de langue N-grammes. 

La fonction ***train_models*** prend en entrée une liste de titres de chansons et construit trois modèles: unigramme, bigramme et trigramme.

Les modèles sont conservés dans ***models***, un dictionnaire Python qui prend la forme 

<pre>
{
   1: modele_unigramme, 
   2: modele_bigramme, 
   3: modele_trigramme
}
</pre>

avec comme clé la valeur N du modèle et comme valeur le modèle construit par NLTK.

In [8]:
from nltk import word_tokenize

In [9]:
import ssl
import nltk

try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass

print(nltk.download('punkt'), nltk.download('punkt_tab'))

True True


[nltk_data] Downloading package punkt to /Users/anto.crgn/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/anto.crgn/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
models = {} 

In [11]:
from nltk.lm.models import Laplace

In [12]:
from nltk.util import pad_sequence ,ngrams

BOS = '<BOS>'
EOS = '<EOS>'

In [13]:
def build_vocabulary(text_list):
    all_unigrams = list()
    for sentence in text_list:
        word_list = word_tokenize(sentence.lower())
        all_unigrams = all_unigrams + word_list
    voc = set(all_unigrams)
    voc.add(BOS)
    voc.add(EOS)
    return list(voc)

In [14]:
def get_ngrams(text_list, n=2):
    all_ngrams = list()
    for sentence in text_list:
        tokens = word_tokenize(sentence.lower())
        padded_sent = list(pad_sequence(tokens, pad_left=True, left_pad_symbol=BOS,
                                        pad_right=True, right_pad_symbol=EOS, n=n))
        all_ngrams = all_ngrams + list(ngrams(padded_sent, n=n))
    return all_ngrams

In [15]:
def train_models(texts):
    """ Vous ajoutez à partir d'ici le code dont vous avez besoin
        pour construire les différents modèles N-grammes.
        Cette fonction doit construire tous les modèles en une seule passe.
    """   
    vocabulary = build_vocabulary(texts)
    for n in range(1,4):
        corpus_ngrams = get_ngrams(texts, n=n)
        model = Laplace(n)
        model.fit([corpus_ngrams], vocabulary_text=vocabulary)
        models[n] = model


In [16]:
train_models(titles)
print(models)

{1: <nltk.lm.models.Laplace object at 0x15348c460>, 2: <nltk.lm.models.Laplace object at 0x1530f4490>, 3: <nltk.lm.models.Laplace object at 0x1530f42e0>}


In [17]:
print(len(models[1].vocab))
print("love" in models[1].vocab)

441
True


### Vérifications des modèles

Ordre de chaque modèle, taille du vocabulaire et découpage d'un même titre selon l'ordre N (padding `<BOS>` / `<EOS>`).

In [18]:
# Ordre des modèles et taille du vocabulaire
print({n: models[n].order for n in models})
print("Taille du vocabulaire V :", len(models[1].vocab))

# Découpage d'un titre selon l'ordre du modèle
for n in (1, 2, 3):
    print(n, get_ngrams(["let it be"], n=n))

{1: 1, 2: 2, 3: 3}
Taille du vocabulaire V : 441
1 [('let',), ('it',), ('be',)]
2 [('<BOS>', 'let'), ('let', 'it'), ('it', 'be'), ('be', '<EOS>')]
3 [('<BOS>', '<BOS>', 'let'), ('<BOS>', 'let', 'it'), ('let', 'it', 'be'), ('it', 'be', '<EOS>'), ('be', '<EOS>', '<EOS>')]


## Section 3 - Estimation - Choix parmi plusieurs candidats

In [19]:
def select_candidate(candidates, n=3):
    """ 
        Cette fonction retourne le candidat retenu ainsi que son score attribué par le modèle.
        
        *candidates* est la liste de passages de textes obtenu à partir du fichier de test. 
        Un seul des passage est un titre de chanson connu. 
        
        Le paramètre n désigne le modèle utilisé.
        1 - unigramme NLTK, 2 - bigramme NLTK, 3 - trigramme NLTK
         
    """

    best_score = float("inf")
    result = None
    for candidate in candidates:
        ngrams_cand = get_ngrams([candidate], n=n)
        score = models[n].perplexity(ngrams_cand)
        if score < best_score:
            best_score = score
            result = candidate
    return result, best_score


In [20]:
print(select_candidate(["i want to hold and love","i want to hold something", "i want to hold your hand"], n=2))

('i want to hold your hand', 74.80783323340664)


### Exemple : perplexité des trois modèles sur un test

Plus la perplexité est basse, plus le candidat paraît vraisemblable au modèle. `select_candidate` retient donc le candidat de perplexité minimale.

In [21]:
candidats = ["let it be", "let me be", "let it roll"]
tableau = pd.DataFrame(
    {f"{n}-gramme": [round(models[n].perplexity(get_ngrams([c], n=n)), 1) for c in candidats]
     for n in (1, 2, 3)},
    index=candidats)
display(tableau)
for n in (1, 2, 3):
    print(f"{n}-gramme choisit :", select_candidate(candidats, n=n))

,1-gramme,2-gramme,3-gramme
let it be,119.6,131.6,143.6
let me be,72.0,90.3,117.8
let it roll,217.3,155.6,189.4


1-gramme choisit : ('let me be', 71.96956554606042)
2-gramme choisit : ('let me be', 90.33235747198427)
3-gramme choisit : ('let me be', 117.76905054969343)


### Évaluation sur les 20 tests d'estimation

In [22]:
for n in range(1, 4):
    correct = 0
    for test in tests_estimation:
        choix, score = select_candidate(test["Candidates"], n=n)
        if choix == test["Song"]:
            correct = correct + 1
    print("Modèle", n, ":", correct, "/", len(tests_estimation))

Modèle 1 : 6 / 20
Modèle 2 : 10 / 20
Modèle 3 : 13 / 20


### Détail par test (✓ = bon titre retrouvé)

In [23]:
detail = pd.DataFrame(
    {f"{n}-gramme": ["✓" if select_candidate(t["Candidates"], n=n)[0] == t["Song"] else "✗"
                     for t in tests_estimation]
     for n in (1, 2, 3)},
    index=[t["Song"] for t in tests_estimation])
display(detail)

,1-gramme,2-gramme,3-gramme
i want to hold your hand,✗,✓,✓
let it be,✗,✗,✗
i want to know what love is,✗,✗,✗
i will survive,✗,✗,✗
i will wait,✓,✓,✓
i gotta feeling,✗,✗,✗
i want to break free,✗,✓,✓
we will rock you,✗,✗,✗
i kissed a girl,✓,✓,✓
i believe i can fly,✗,✗,✓


### Résultats de la sous-tâche 1 (estimation)

| Modèle | Bonnes réponses | Précision |
|---|---|---|
| Unigramme | 6 / 20 | 30 % |
| Bigramme | 10 / 20 | 50 % |
| Trigramme | 13 / 20 | 65 % |

La précision augmente avec la taille du contexte : l'unigramme ne regarde que la fréquence des mots isolés, alors que le bigramme et le trigramme tiennent compte des enchaînements propres aux vrais titres.

Les erreurs restantes viennent surtout du lissage de Laplace, trop fort pour un petit corpus (572 titres, vocabulaire de 441 mots) : les comptes réellement observés sont écrasés par le +1 ajouté à chaque mot du vocabulaire. Par exemple, « let me be » est préféré à « let it be » par les trois modèles, alors que « let it be » figure dans le corpus.

L'analyse détaillée est présentée dans le rapport (section 1.1).

## Section 4 - Génération - Compléter les titres de chansons

In [24]:
import random


# Distinguer les vrais mots (pas des symbole)
def is_real_word(model, word):
    return word not in (BOS, EOS, model.vocab.unk_label)

# Ajoute N-1 symboles <BOS> devant le titre incomplet, comme lors de l'entraînement.
def padded_context(model, tokenized_seed):
    return [BOS] * (model.order - 1) + list(tokenized_seed)


# chercher le context à donner au model 
def known_history(model, context):
    n_hist = model.order - 1
    history = list(context[len(context) - n_hist:]) if n_hist > 0 else []
    while True:
        successors = model.context_counts(model.vocab.lookup(history))
        if any(is_real_word(model, w) for w in successors):
            return history
        if not history:
            return None
        history = history[1:]  


# generation du prochain mot suivant le context
def next_word(model, context, rand_gen):
    history = known_history(model, context)
    if history is None:                # aucun contexte connu : on repart comme au debut d'un titre
        history = [BOS] * (model.order - 1)

    word = model.generate(1, text_seed=history, random_seed=rand_gen)
    while not is_real_word(model, word): 
        word = model.generate(1, text_seed=history, random_seed=rand_gen)
    return word

In [25]:
# Entrainement et Verification si les models sont dans le bon ordre pour les utiliser a la generation
train_models(titles)
print({n: models[n].order for n in models})

{1: 1, 2: 2, 3: 3}


In [26]:
def complete_title(model, tokenized_seed, nb_words, random_seed=46):
    """
        Génère des mots et les ajoute à la fin du texte tokenisé. 
        Retourne la liste de mots du titre complet (c.-à-d. le *tokenized_seed* + les *nb_words* générés).
        
        model: Le modèle N-gramme utilisé
        tokenized_seed: le titre incomplet, sans les masques, qui a été tokenisé. Par exemple: ["le", "soleil"].
        random_seed: utilisé par NLTK pour la génération de nombre aléatoire. 
        
    """    
    
    rand_gen = random.Random(random_seed)
    context = padded_context(model, tokenized_seed)
    new_words = []
    for _ in range(nb_words):
        word = next_word(model, context, rand_gen)
        new_words.append(word)
        context.append(word)  # le mot généré devient le contexte du suivant

    generated = list(tokenized_seed) + new_words
    return generated

# des prints pour tester notre model
print("******** 1_Gramme ********")
print(complete_title(models[1], ["i", "want", "to"], 2))
print(complete_title(models[1], ["i", "believe", "i"], 2))
print(complete_title(models[1], ["mot", "inconnu"], 2))   # contexte jamais vu

print("******** 2_Gramme ********")
print(complete_title(models[2], ["i", "want", "to"], 2))
print(complete_title(models[2], ["i", "believe", "i"], 2))
print(complete_title(models[2], ["mot", "inconnu"], 2))   # contexte jamais vu

print("******** 3_Gramme ********")
print(complete_title(models[3], ["i", "want", "to"], 2))
print(complete_title(models[3], ["i", "believe", "i"], 2))
print(complete_title(models[3], ["mot", "inconnu"], 2))   # contexte jamais vu

******** 1_Gramme ********
['i', 'want', 'to', 'want', 'know']
['i', 'believe', 'i', 'want', 'know']
['mot', 'inconnu', 'want', 'know']
******** 2_Gramme ********
['i', 'want', 'to', 'tomorrow', 'let']
['i', 'believe', 'i', 'will', 'go']
['mot', 'inconnu', 'we', 'belong']
******** 3_Gramme ********
['i', 'want', 'to', 'want', 'me']
['i', 'believe', 'i', 'can', 'fly']
['mot', 'inconnu', 'we', 'let']


In [27]:
# Taille du vocabulaire V (identique pour les trois modèles)
V = len(models[1].vocab)
print("V (taille du vocabulaire) :", V)        

# ---------- Unigramme ----------
unigram_counts = models[1].counts.unigrams
N = sum(unigram_counts[w] for w in unigram_counts if w not in (BOS, EOS))
print("\nN (nombre total de mots) :", N)                             
print("Dénominateur N + V       :", N + V)                           
print("C(want) :", unigram_counts["want"], "-> P_Laplace =", round((unigram_counts["want"] + 1) / (N + V), 4))
print("C(know) :", unigram_counts["know"], "-> P_Laplace =", round((unigram_counts["know"] + 1) / (N + V), 4))

# ---------- Bigramme - titre 2 : i believe i *** *** ----------
bigram_counts = models[2].counts[2]
c_i = bigram_counts[("i",)].N()
c_i_will = bigram_counts[("i",)]["will"]
c_will = bigram_counts[("will",)].N()
c_will_go = bigram_counts[("will",)]["go"]
print(f"\nC(i) = {c_i}, C(i will) = {c_i_will} -> P_Laplace(will | i) = {(c_i_will + 1) / (c_i + V):.4f}")
print(f"C(will) = {c_will}, C(will go) = {c_will_go} -> P_Laplace(go | will) = {(c_will_go + 1) / (c_will + V):.4f}")

# ---------- Trigramme ----------
trigram_counts = models[3].counts[3]

debut = trigram_counts[(BOS, BOS)]
print("Premiers mots distincts :", len(debut)) 
print("mots de debut du plus fréquent au moins fréquent")
print(debut.most_common())  

# P_Laplace(w | w1 w2) = (C(w1 w2 w) + 1) / (C(w1 w2) + V)
def trigram_prob(w1, w2, w):
    c_h = trigram_counts[(w1, w2)].N()
    c_hw = trigram_counts[(w1, w2)][w]
    print(f"C({w1} {w2}) = {c_h}, C({w1} {w2} {w}) = {c_hw} -> P_Laplace({w} | {w1} {w2}) = {(c_hw + 1) / (c_h + V):.4f}")

print("\nTitre 2 : i believe i *** ***")
trigram_prob("believe", "i", "can")
trigram_prob("i", "can", "fly")

print("\nTitre 1 : i want to *** ***")
trigram_prob("want", "to", "break")
trigram_prob("want", "to", "want")
trigram_prob("to", "want", "me")
print("Successeurs de 'want to' :", dict(trigram_counts[("want", "to")]))

print("\nTitre 3 : début de titre (<BOS> <BOS>)")
trigram_prob(BOS, BOS, "we")
print("C(<BOS> <BOS>) = nombre de titres :", trigram_counts[(BOS, BOS)].N())

V (taille du vocabulaire) : 441

N (nombre total de mots) : 2143
Dénominateur N + V       : 2584
C(want) : 24 -> P_Laplace = 0.0097
C(know) : 5 -> P_Laplace = 0.0023

C(i) = 39, C(i will) = 7 -> P_Laplace(will | i) = 0.0167
C(will) = 10, C(will go) = 1 -> P_Laplace(go | will) = 0.0044
Premiers mots distincts : 29
mots de debut du plus fréquent au moins fréquent
[('hold', 45), ('you', 32), ('let', 27), ('dream', 26), ('i', 26), ('come', 25), ('stand', 25), ('the', 24), ('wake', 24), ('do', 24), ('fly', 24), ('tell', 23), ('take', 23), ('waiting', 23), ('break', 22), ('love', 21), ('we', 21), ('my', 21), ('reach', 20), ('your', 19), ('give', 19), ('run', 19), ('want', 8), ('need', 8), ('wait', 7), ('running', 7), ('holding', 6), ('breaking', 2), ('dreamweaver', 1)]

Titre 2 : i believe i *** ***
C(believe i) = 1, C(believe i can) = 1 -> P_Laplace(can | believe i) = 0.0045
C(i can) = 1, C(i can fly) = 1 -> P_Laplace(fly | i can) = 0.0045

Titre 1 : i want to *** ***
C(want to) = 7, C(want

In [28]:
# separer un exemple de test en : titre incomplet tokenisé, nombre de masques, mots attendus.
def testing(example):
    """  """
    masked = example["Masked"]                                     # "i believe i *** ***"
    nb_masks = masked.count("***")                                 # 2
    seed = word_tokenize(masked.replace("***", "").lower())        # ["i", "believe", "i"]
    expected = word_tokenize(example["Song"].lower())[-nb_masks:]  # ["can", "fly"]
    return seed, nb_masks, expected

# Calcul de la proportion des mots générés qui correspondent au titre connu (mot expected)
def evaluate_generation(model, tests, random_seed=46):
    """ . """
    nb_correct = 0
    nb_total = 0
    rows = []
    for example in tests:
        seed, nb_masks, expected = testing(example)
        completed = complete_title(model, seed, nb_masks, random_seed)
        generated = completed[-nb_masks:]

        correct = sum(g == e for g, e in zip(generated, expected))
        nb_correct += correct
        nb_total += nb_masks
        rows.append({"Masked": example["Masked"],
                     "Généré": " ".join(completed),
                     "Song": example["Song"],
                     "Corrects": f"{correct}/{nb_masks}"})
    return nb_correct / nb_total, pd.DataFrame(rows)

In [29]:
# details sur nos donnees de test
nb_titres = len(tests_generation)
nb_masques = sum(example["Masked"].count("***") for example in tests_generation)

print("Nombre de titres      :", nb_titres) 
print("Nombre de mots masqués :", nb_masques) 

# Appliquer les evaluations pour notre data de test pour la generation et comparer l'exactitude de chaque model
results = []
for n in (1, 2, 3):
    accuracy, _ = evaluate_generation(models[n], tests_generation)
    results.append({"Model": f"{n}-gramme", "Accuracy": round(accuracy, 3)})
display(pd.DataFrame(results))

Nombre de titres      : 40
Nombre de mots masqués : 66


,Model,Accuracy
0,1-gramme,0.000
1,2-gramme,0.227
2,3-gramme,0.636


## Section 5 - Partie réservée pour faire nos tests lors de la correction

Merci de ne pas modifier ni retirer cette section du notebook.  